# Meta Analysis Workflow
Notebook for loading metadata, searching climate data, and finding the nearest weather station by address.

In [1]:
from pathlib import Path

import pandas as pd

from simultaneousness_analysis.meta import MetaSearch, MetaSearchResult, MetaTable
from simultaneousness_analysis.retrieve_data_from_dwd_cdc import main as retrieve_data

## Optional: Retrieve DWD CDC Data

In [2]:
RETRIEVE_DATA = False  # Set to True to download DWD CDC data

if RETRIEVE_DATA:
    print("Retrieving DWD CDC data...")
    retrieve_data()
    print("Data retrieval complete!")

## Run Meta Analysis

In [8]:
data_path = Path.cwd().parent / "data" / "cdc" / "raw"
meta_table = MetaTable(data_path=data_path)
meta_table.table.head()

,type,resolution_value,resolution_unit,measurand,from_date,to_date,stations_id,format,measurand_names,altitude,latitude,longitude,station_name,federal_state
path,,,,,,,,,,,,,,
c:\Users\Anwender\Documents\Development\simultaneousness-analysis\data\cdc\raw\air_temperature\produkt_zehn_min_tu_19910417_19991231_04466.txt,produkt,zehn,min,tu,1991-04-17,1999-12-31,4466,.txt,air temperature,43,54.5275,9.5487,Schleswig,Schleswig-Holstein
c:\Users\Anwender\Documents\Development\simultaneousness-analysis\data\cdc\raw\air_temperature\produkt_zehn_min_tu_19941102_19991231_05930.txt,produkt,zehn,min,tu,1994-11-02,1999-12-31,5930,.txt,air temperature,1,54.6410,10.0238,Schönhagen (Ostseebad),Schleswig-Holstein
c:\Users\Anwender\Documents\Development\simultaneousness-analysis\data\cdc\raw\air_temperature\produkt_zehn_min_tu_19951201_19991231_03032.txt,produkt,zehn,min,tu,1995-12-01,1999-12-31,3032,.txt,air temperature,25,55.0110,8.4125,List auf Sylt,Schleswig-Holstein
c:\Users\Anwender\Documents\Development\simultaneousness-analysis\data\cdc\raw\air_temperature\produkt_zehn_min_tu_19960531_19991231_05516.txt,produkt,zehn,min,tu,1996-05-31,1999-12-31,5516,.txt,air temperature,3,54.5283,11.0606,Fehmarn,Schleswig-Holstein
c:\Users\Anwender\Documents\Development\simultaneousness-analysis\data\cdc\raw\air_temperature\produkt_zehn_min_tu_19960701_19991231_03086.txt,produkt,zehn,min,tu,1996-07-01,1999-12-31,3086,.txt,air temperature,15,53.8025,10.6989,Lübeck-Blankensee,Schleswig-Holstein


In [9]:
meta_table.export(suffix="json")
meta_table.export(suffix="xlsx")
print("Meta analysis exported to JSON and XLSX formats")

Meta analysis exported to JSON and XLSX formats


## Search Meta Table

In [10]:
search_param = MetaSearch(
    measurand_names=["solar radiation"],
    to_date=pd.Timestamp(year=2000, month=1, day=1),
    stations_id=[4466],
)

search_result: MetaSearchResult = meta_table.search(search_param=search_param)
print(f"{search_result.paths=}")
print(f"{search_result.length=}")

search_result.paths=[WindowsPath('c:/Users/Anwender/Documents/Development/simultaneousness-analysis/data/cdc/raw/solar/produkt_zehn_min_sd_19910417_19991231_04466.txt')]
search_result.length=1


## Find Nearest Station by Address

In [ ]:
address = {
    "street": "Rathausplatz",
    "house_number": "1",
    "zip_code": "24103",
    "city": "Kiel",
}
nearest_station = meta_table.get_nearest_station_id_by_address(**address)
print(
    f"Nearest station for Kiel address: {nearest_station.stations_id} "
    f"({nearest_station.distance_km:.2f} km)",
)
station_info = meta_table.get_station_info(nearest_station.stations_id)
print(
    f"Nearest station for Kiel address in {nearest_station.distance_km:.2f} km: {station_info}",
)


Nearest station for Kiel address: 2564 (6.13 km)
Nearest station for Kiel address in 6.13 km: MetaStationInfo(stations_id=2564, station_name='Kiel-Holtenau', federal_state='Schleswig-Holstein', latitude=54.3776, longitude=10.1424, altitude=28.0)


# Find nearest station by coordinates

In [13]:
# Coordinates for Bregning-West 4a, 24975 Husby
latitude = 54.769328
longitude = 9.568905
nearest_station_coords = meta_table.get_nearest_station_id_by_coordinates(
    latitude=latitude,
    longitude=longitude,
)
print(
    f"Nearest station for coordinates ({latitude:.7f}, {longitude:.7f}): "
    f"{nearest_station_coords.stations_id} "
    f"({nearest_station_coords.distance_km:.2f} km)",
)
station_info_coords = meta_table.get_station_info(nearest_station_coords.stations_id)
print(
    f"Nearest station for coordinates in {nearest_station_coords.distance_km:.2f} km: {station_info_coords}",
)

Nearest station for coordinates (54.7693280, 9.5689050): 1379 (12.43 km)
Nearest station for coordinates in 12.43 km: MetaStationInfo(stations_id=1379, station_name='Flensburg (Schäferhaus)', federal_state='Schleswig-Holstein', latitude=54.7737, longitude=9.3753, altitude=41.0)
